In [3]:
# !pip install gensim
# !pip install biopython
# need pip install scikit-learn==0.23.1
# pip install numpy==1.19.5
# pip install pandas==1.3.5
import pandas as pd

#DO NOT CHANGE ANYTHING IN THIS CELL. MOVE ON TO THE FOLLOWING ONE TO GET THE PREDICTION.
import os
os.environ['PATH'] = "/Users/newuser/ncbi-blast-2.16.0+/bin:" + os.environ['PATH']

import numpy as np
from gensim.models import word2vec

class ProtVec(word2vec.Word2Vec):

    def __init__(self, fasta_fname=None, corpus=None, n=3, size=100, corpus_fname="corpus.txt",  sg=1, window=25, min_count=1, workers=20):
        """
        Either fname or corpus is required.
        fasta_fname: fasta file for corpus
        corpus: corpus object implemented by gensim
        n: n of n-gram
        corpus_fname: corpus file path
        min_count: least appearance count in corpus. if the n-gram appear k times which is below min_count, the model does not remember the n-gram
        """

        self.n = n
        self.size = size
        self.fasta_fname = fasta_fname

        if corpus is None and fasta_fname is None:
            raise Exception("Either fasta_fname or corpus is needed!")

        if fasta_fname is not None:
            print('Generate Corpus file from fasta file...')
            generate_corpusfile(fasta_fname, n, corpus_fname)
            corpus = word2vec.Text8Corpus(corpus_fname)

        word2vec.Word2Vec.__init__(self, corpus, size=size, sg=sg, window=window, min_count=min_count, workers=workers)

    def to_vecs(self, seq):
        """
        convert sequence to three n-length vectors
        e.g. 'AGAMQSASM' => [ array([  ... * 100 ], array([  ... * 100 ], array([  ... * 100 ] ]
        """
        ngram_patterns = split_ngrams(seq, self.n)

        protvecs = []
        for ngrams in ngram_patterns:
            ngram_vecs = []
            for ngram in ngrams:
                try:
                    ngram_vecs.append(self.wv[ngram])
                except:
                    raise Exception("Model has never trained this n-gram: " + ngram)
            protvecs.append(sum(ngram_vecs))
        return protvecs
    
    
    def get_vector(self, seq):
        """
        sum and normalize the three n-length vectors returned by self.to_vecs
        """
        #return normalize(sum(self.to_vecs(seq)))
        return sum(self.to_vecs(seq))

    
def load_protvec(model_fname):
    return word2vec.Word2Vec.load(model_fname)

pv = load_protvec('src/files/DeePhase/__PREDICT/tools/Embeddings/swissprot_size200_window25.model')

SEED = 42
np.random.seed(SEED)

from src.files.DeePhase.__PREDICT.deephase_utils import *

FileNotFoundError: [Errno 2] No such file or directory: '__PREDICT/tools/Embeddings/swissprot_size200_window25.model'

In [ ]:
def extract_deephase_score(seq):
    # # Create a DataFrame with the input sequence
    df = pd.DataFrame({'sequence_final': [seq]})
    
    # Call the DeePhase function (assuming it returns a string)
    deephase_result = DeePhase(df)
    
    # Extract the score from the result (assuming it's the last element after splitting)
    # score_str = deephase_result.split()[-1]
    
    # # Create a new DataFrame with the sequence and the score
    # new_df = pd.DataFrame({
    #     'sequence_final': [seq],  # Add the sequence
    #     'deephase_score': [score_str]  # Add the score
    # })
    
    # # Print the result and score for debugging
    # print(deephase_result)
    # print(score_str)
    
    # Return the new DataFrame
    return deephase_result

In [3]:
idr_seq = "MESNHKSGDGLSGTQKEAALRALVQRTGYSLVQENGQRKYGGPPPGWDAAPPERGCEIFIGKLPRDLFEDELIPLCEKIGKIYEMRMMMDFNGNNRGYAFVTFSNKVEAKNAIKQLNNYEIRNGRLLGVCASVDNCRLFVGGIPKTKKREEILSEMKKVTEGVVDVIVYPSAADKTKNRGFAFVEYESHRAAAMARRKLLPGRIQLWGHGIAVDWAEPEVEVDEDTMSSVKILYVRNLMLSTSEEMIEKEFNNIKPGAVERVKKIRDYAFVHFSNREDAVEAMKALNGKVLDGSPIEVTLAKPVDKDSYVRYTRGTGGRGTMLQGEYTYSLGQVYDPTTTYLGAPVFYAPQTYAAIPSLHFPATKGHLSNRAIIRAPSVREIYMNVPVGAAGVRGLGGRGYLAYTGLGRGYQVKGDKREDKLYDILPGMELTPMNPVTLKPQGIKLAPQILEEICQKNNWGQPVYQLHSAIGQDQRQLFLYKITIPALASQNPAIHPFTPPKLSAFVDEAKTYAAEYTLQTLGIPTDGGDGTMATAAAAATAFPGYAVPNATAPVSAAQLKQAVTLGQDLAAYTTYEVYPTFAVTARGDGYGTF"

extract_deephase_score(idr_seq)

/Users/newuser/anaconda3/envs/1433predictor20250101/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/newuser/anaconda3/envs/1433predictor20250101/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/newuser/1433predictor/src/files/DeePhase/__PREDICT/deephase_utils.py:391: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_interm['phys_multi'] = predict_multiclass('phys_multi', data_phys_sel)['prediction_phys_multi']
/Users/newuser/anaconda3/envs/1433predictor20250101/lib/python3.9/site-packages/sklearn/b

(0.237, 0.529, 0.383)

In [4]:
import csv

# Define the sequence
sequence = "MSSQSHPDGLSGRDQPVELLNPARVNHMPSTVDVATALPLQVAPTAVPMDLRLDHQFSLPLEPALREQQLQQELLALKQKQQIQRQILIAEFQRQHEQLSRQHEAQLHEHIKQQQEM"

# Define the name of the CSV file
csv_filename = "input_seqs.csv"

# Create and write to the CSV file
with open(csv_filename, mode='w', newline='') as csvfile:
    # Initialize the CSV writer
    csvwriter = csv.writer(csvfile)
    
    # Write the header
    csvwriter.writerow(['seq'])
    
    # Write the sequence
    csvwriter.writerow([sequence])

print(f"CSV file '{csv_filename}' created successfully.")

CSV file 'input_seqs.csv' created successfully.


In [5]:
def main():

    # Load the CSV file from the current directory
    input_csv_file = "input_seqs.csv"
    dataframe = pd.read_csv(input_csv_file)

    # Ensure the 'seq' column exists
    if 'seq' not in dataframe.columns:
        raise ValueError("CSV file must contain a column named 'seq'")

    # Create a new column 'deephase_score' by applying the scoring function
    dataframe['deephase_score'] = dataframe['seq'].apply(extract_deephase_score)

    # Save the modified dataframe to a new CSV file with the results
    output_csv_file = "output_with_deephase_score.csv"
    dataframe.to_csv(output_csv_file, index=False)

    print(f"Processed results saved to {output_csv_file}")

In [6]:
main()

Processed results saved to output_with_deephase_score.csv


/Users/newuser/anaconda3/envs/1433predictor20250101/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/newuser/anaconda3/envs/1433predictor20250101/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/newuser/1433predictor/src/files/DeePhase/__PREDICT/deephase_utils.py:391: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_interm['phys_multi'] = predict_multiclass('phys_multi', data_phys_sel)['prediction_phys_multi']
/Users/newuser/anaconda3/envs/1433predictor20250101/lib/python3.9/site-packages/sklearn/b

In [7]:
# extra test of the deephase score for the deployed score distribution